<a href="https://colab.research.google.com/github/elsa-paul11/de-portfolio-2026/blob/main/module-01-storage/notebooks/m1_d1_s3_parquet_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Cell 1
!pip install pyspark -q

In [5]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.1 MB/s eta 0:00:00


In [6]:

import logging
import sys
import os
import random
from datetime import date
from decimal import Decimal

import boto3
from google.colab import userdata
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, StringType, DateType,
    DecimalType, StructType, StructField
)

In [ ]:

os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = "ap-south-1"

In [10]:
# Instead of random print() statements, every message has a timestamp + level
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    stream=sys.stdout
)
logger = logging.getLogger("module1")

logger.info("Logger working correctly")
#The logging module was already configured by Colab internally before your cell ran, so basicConfig silently did nothing.


In [8]:
# Force reconfigure logging
logger = logging.getLogger("module1")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    ))
    logger.addHandler(handler)

logger.propagate = False  # Prevent Colab's root logger from interfering

logger.info("Logger working correctly")

2026-05-15 17:57:19,954 | INFO | Logger working correctly


In [9]:
spark = (
    SparkSession.builder
    .appName("m1_orders_pipeline")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")  # Suppress Spark's own noisy logs
logger.info(f"Spark started | version={spark.version}")

2026-05-15 17:57:42,070 | INFO | Spark started | version=4.0.2


In [10]:
# NEVER let Spark guess your schema in production
# inferSchema=True reads your entire file just to guess column types
# At 10TB that costs time and money before your job even starts

ORDERS_SCHEMA = StructType([
    StructField("order_id",     IntegerType(),      nullable=False),
    StructField("customer_id",  IntegerType(),      nullable=False),
    StructField("product_id",   IntegerType(),      nullable=False),
    StructField("order_date",   DateType(),         nullable=False),
    StructField("amount",       DecimalType(10,2),  nullable=False),
    StructField("status",       StringType(),       nullable=True),
    StructField("region",       StringType(),       nullable=True),
])

logger.info("Schema defined")
print(ORDERS_SCHEMA)

2026-05-15 17:57:45,345 | INFO | Schema defined
StructType([StructField('order_id', IntegerType(), False), StructField('customer_id', IntegerType(), False), StructField('product_id', IntegerType(), False), StructField('order_date', DateType(), False), StructField('amount', DecimalType(10,2), False), StructField('status', StringType(), True), StructField('region', StringType(), True)])


In [13]:


random.seed(42)

statuses = ["COMPLETED", "PENDING", "CANCELLED", "REFUNDED", None]
regions  = ["IN-SOUTH", "IN-NORTH", "IN-WEST", "IN-EAST"]

rows = []
for i in range(1, 10_001):
    rows.append((
        i,
        random.randint(1, 5000),
        random.randint(1, 500),
        date(random.randint(2023, 2024), random.randint(1, 12), random.randint(1, 28)),
        Decimal(str(round(random.uniform(10.0, 5000.0), 2))),  # ← wrapped in Decimal()
        random.choice(statuses),
        random.choice(regions),
    ))

raw_df = spark.createDataFrame(rows, schema=ORDERS_SCHEMA)

logger.info(f"Data created | rows={raw_df.count():,}")
raw_df.show(5)
raw_df.printSchema()

2026-05-15 18:00:04,060 | INFO | Data created | rows=10,000
+--------+-----------+----------+----------+-------+---------+--------+
|order_id|customer_id|product_id|order_date| amount|   status|  region|
+--------+-----------+----------+----------+-------+---------+--------+
|       1|        913|        13|2024-04-08| 706.29|COMPLETED|IN-SOUTH|
|       2|       4838|       217|2023-01-03|1101.00|     NULL|IN-SOUTH|
|       3|       4598|       102|2024-04-15|2950.44|COMPLETED|IN-NORTH|
|       4|       3463|       175|2024-03-07|4786.49|CANCELLED|IN-SOUTH|
|       5|        760|       195|2023-06-28|1726.34|CANCELLED|IN-SOUTH|
+--------+-----------+----------+----------+-------+---------+--------+
only showing top 5 rows
root
 |-- order_id: integer (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- product_id: integer (nullable = false)
 |-- order_date: date (nullable = false)
 |-- amount: decimal(10,2) (nullable = false)
 |-- status: string (nullable = true)
 |-- re

In [14]:
# Partitioning = creating folders on S3 by date
# s3://bucket/orders/event_year=2024/event_month=01/file.parquet
#
# Why? A query for Jan 2024 opens ONLY that folder
# Without partitioning: Spark opens ALL files to find Jan 2024 data
# With partitioning: Spark skips everything except Jan 2024 folder

transformed_df = (
    raw_df
    .withColumn("event_year",  F.year("order_date").cast("string"))
    .withColumn("event_month", F.lpad(F.month("order_date").cast("string"), 2, "0"))
    .withColumn("ingested_at", F.current_timestamp())  # audit trail
)

logger.info("Partition columns added")
transformed_df.select("order_id", "order_date", "event_year", "event_month", "ingested_at").show(5)

2026-05-15 18:00:08,969 | INFO | Partition columns added
+--------+----------+----------+-----------+--------------------+
|order_id|order_date|event_year|event_month|         ingested_at|
+--------+----------+----------+-----------+--------------------+
|       1|2024-04-08|      2024|         04|2026-05-15 18:00:...|
|       2|2023-01-03|      2023|         01|2026-05-15 18:00:...|
|       3|2024-04-15|      2024|         04|2026-05-15 18:00:...|
|       4|2024-03-07|      2024|         03|2026-05-15 18:00:...|
|       5|2023-06-28|      2023|         06|2026-05-15 18:00:...|
+--------+----------+----------+-----------+--------------------+
only showing top 5 rows


In [16]:
# CELL 8A — Write Parquet to local /tmp/
LOCAL_PATH = "/tmp/orders_output"

(
    transformed_df
    .coalesce(4)
    .write
    .mode("overwrite")
    .partitionBy("event_year", "event_month")
    .option("compression", "snappy")
    .parquet(LOCAL_PATH)
)

logger.info(f"Local write complete | path={LOCAL_PATH}")

# Confirm files exist locally
import os
for root, dirs, files in os.walk(LOCAL_PATH):
    for file in files:
        print(os.path.join(root, file))

2026-05-15 18:02:25,623 | INFO | Local write complete | path=/tmp/orders_output
/tmp/orders_output/_SUCCESS
/tmp/orders_output/._SUCCESS.crc
/tmp/orders_output/event_year=2023/event_month=02/part-00000-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet
/tmp/orders_output/event_year=2023/event_month=02/.part-00000-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet.crc
/tmp/orders_output/event_year=2023/event_month=02/.part-00001-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet.crc
/tmp/orders_output/event_year=2023/event_month=02/part-00001-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet
/tmp/orders_output/event_year=2023/event_month=05/part-00000-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet
/tmp/orders_output/event_year=2023/event_month=05/.part-00000-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet.crc
/tmp/orders_output/event_year=2023/event_month=05/.part-00001-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet.crc
/tmp/orders_o

In [20]:

s3 = boto3.client("s3", region_name="ap-south-1")
BUCKET = "elsa-de-prep-2026"
S3_PREFIX = "bronze/orders"
LOCAL_PATH = "/tmp/orders_output"

uploaded = 0
for root, dirs, files in os.walk(LOCAL_PATH):
    for filename in files:
        local_file_path = os.path.join(root, filename)
        relative_path = os.path.relpath(local_file_path, LOCAL_PATH)
        s3_key = f"{S3_PREFIX}/{relative_path}"

        s3.upload_file(local_file_path, BUCKET, s3_key)
        print(f"Uploaded: {s3_key}")
        uploaded += 1

print(f"\nTotal uploaded: {uploaded}")

Uploaded: bronze/orders/_SUCCESS
Uploaded: bronze/orders/._SUCCESS.crc
Uploaded: bronze/orders/event_year=2023/event_month=02/part-00000-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet
Uploaded: bronze/orders/event_year=2023/event_month=02/.part-00000-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet.crc
Uploaded: bronze/orders/event_year=2023/event_month=02/.part-00001-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet.crc
Uploaded: bronze/orders/event_year=2023/event_month=02/part-00001-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet
Uploaded: bronze/orders/event_year=2023/event_month=05/part-00000-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet
Uploaded: bronze/orders/event_year=2023/event_month=05/.part-00000-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet.crc
Uploaded: bronze/orders/event_year=2023/event_month=05/.part-00001-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet.crc
Uploaded: bronze/orders/event_year=2023/event_mo

In [18]:
# Read back from LOCAL path (S3A driver unavailable in pip PySpark)
df_verify = spark.read.parquet(LOCAL_PATH)

row_count = df_verify.count()
partition_count = df_verify.select("event_year", "event_month").distinct().count()

logger.info(f"Verification | rows={row_count:,} | partitions={partition_count}")

df_verify.groupBy("event_year", "event_month") \
         .count() \
         .orderBy("event_year", "event_month") \
         .show(30)

2026-05-15 18:12:55,913 | INFO | Verification | rows=10,000 | partitions=24
+----------+-----------+-----+
|event_year|event_month|count|
+----------+-----------+-----+
|      2023|          1|  365|
|      2023|          2|  436|
|      2023|          3|  403|
|      2023|          4|  440|
|      2023|          5|  418|
|      2023|          6|  415|
|      2023|          7|  419|
|      2023|          8|  437|
|      2023|          9|  409|
|      2023|         10|  410|
|      2023|         11|  421|
|      2023|         12|  399|
|      2024|          1|  432|
|      2024|          2|  433|
|      2024|          3|  431|
|      2024|          4|  401|
|      2024|          5|  422|
|      2024|          6|  425|
|      2024|          7|  427|
|      2024|          8|  389|
|      2024|          9|  384|
|      2024|         10|  409|
|      2024|         11|  451|
|      2024|         12|  424|
+----------+-----------+-----+



In [21]:
# Verify files landed in S3 using boto3 directly
s3 = boto3.client("s3", region_name="ap-south-1")

response = s3.list_objects_v2(
    Bucket="elsa-de-prep-2026",
    Prefix="bronze/orders/"
)

files = response.get("Contents", [])
logger.info(f"Files in S3: {len(files)}")

for obj in files[:10]:  # Show first 10
    print(f"  {obj['Key']}  ({obj['Size']} bytes)")

2026-05-15 18:15:25,781 | INFO | Files in S3: 98
  bronze/orders/._SUCCESS.crc  (8 bytes)
  bronze/orders/_SUCCESS  (0 bytes)
  bronze/orders/event_year=2023/event_month=01/.part-00000-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet.crc  (56 bytes)
  bronze/orders/event_year=2023/event_month=01/.part-00001-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet.crc  (56 bytes)
  bronze/orders/event_year=2023/event_month=01/part-00000-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet  (6128 bytes)
  bronze/orders/event_year=2023/event_month=01/part-00001-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet  (5808 bytes)
  bronze/orders/event_year=2023/event_month=02/.part-00000-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet.crc  (64 bytes)
  bronze/orders/event_year=2023/event_month=02/.part-00001-f4eb894f-3e3f-4c6e-b66b-29c4b58e5602.c000.snappy.parquet.crc  (60 bytes)
  bronze/orders/event_year=2023/event_month=02/part-00000-f4eb894f-3e3f-4c6e-b66b-29c4b5